# DSDE Election Pipeline V8

Friend's pipeline structure + V8 direct OCR API with custom prompt.

**Run order:**
1. Cell 1 — Install
2. Cell 2 — Config
3. Cell 3 — Helpers
4. Cell 4 — OCR functions
5. Cell 5 — LLM functions
6. Cell 6 — `run_ocr_stage(all_pdfs)`
7. Cell 7 — `run_llm_stage(all_pdfs)`
8. Cell 8 — `write_timing_reports()`

In [1]:
# ============================================================
# CELL 1: Install
# ============================================================
!pip install pymupdf Pillow python-dotenv openai requests -q
print("Done")

Done


In [3]:
# ============================================================
# CELL 2: Config
# ============================================================
import os, re, json, csv, shutil, time, tempfile, threading
from pathlib import Path
from collections import deque
from dotenv import load_dotenv
from openai import OpenAI
import requests as req
from PIL import Image
import fitz   # PyMuPDF
import base64, io

load_dotenv()

TYPHOON_KEY   = os.getenv("TYPHOON_KEY", "")
PROVINCE      = "อุบลราชธานี"
CONSTITUENCY  = 2
RAW_DATA_DIR  = "rawData"
OUTPUT_DIR    = "output_v2"
OCR_TEXT_DIR  = "ocr_texts_v2"

Path(OUTPUT_DIR).mkdir(exist_ok=True)
Path(OCR_TEXT_DIR).mkdir(exist_ok=True)

TYPHOON_BASE_URL  = "https://api.opentyphoon.ai/v1"
OCR_MODEL         = "typhoon-ocr"
LLM_MODEL         = "typhoon-v2.5-30b-a3b-instruct"

OCR_LIMIT_PER_MIN = 20
LLM_LIMIT_PER_MIN = 200
OCR_PAGE_TIMEOUT  = 90
MAX_IMAGE_SIDE    = 2200
MAX_IMAGE_BYTES   = 4_000_000

os.environ["OPENAI_API_KEY"] = TYPHOON_KEY

typhoon_client = OpenAI(base_url=TYPHOON_BASE_URL, api_key=TYPHOON_KEY)

VALID_PARTIES_TEXT = """
ประชาธิปัตย์, ประชากรไทย, ความหวังใหม่, เพื่อไทย, ภูมิใจไทย,
สังคมประชาธิปไตยไทย, รักชาติ, ประชาธิปไตยใหม่, ครูไทยเพื่อประชาชน,
ประชาชน, ไทยก้าวใหม่, เสรีรวมไทย, พลังไทยรักชาติ, เพื่อชีวิตใหม่,
ทางเลือกใหม่, เศรษฐกิจ, สร้างอนาคตไทย, พลังธรรมใหม่, ไทยธรรม,
ไทยพร้อม, ปวงชนไทย, เพื่อชาติไทย, ประชาชาติ, แผ่นดินธรรม, คลองไทย,
พลังประชารัฐ, เป็นธรรม, พลังเพื่อไทย, ประชาไทย, กรีน, วิชชั่นใหม่,
พลวัต, กล้าธรรม, ไทยรวมไทย, ฟิวชัน, พลังสังคมใหม่, ไทยสร้างไทย,
รวมไทยสร้างชาติ, มิติใหม่, ไทยภักดี, ไทยพิทักษ์ธรรม, ไทยชนะ,
ไทรวมพลัง, ก้าวอิสระ, โอกาสใหม่, ท้องที่ไทย, ใหม่, แรงงานสร้างชาติ, ไทยก้าวหน้า,
พร้อม, รวมใจไทย, ประชาอาสาชาติ, ไทยทรัพย์ทวี, รวมพลังประชาชน,
เพื่อบ้านเมือง, อนาคตไทย, เครือข่ายชาวนาแห่งประเทศไทย
"""

# Runtime timing logs
OCR_REQUEST_TIMINGS = []
OCR_FILE_TIMINGS    = []
LLM_REQUEST_TIMINGS = []
LLM_FILE_TIMINGS    = []

print(f"TYPHOON_KEY : {'set' if TYPHOON_KEY else 'MISSING'}")
print(f"OCR_MODEL   : {OCR_MODEL}")
print(f"LLM_MODEL   : {LLM_MODEL}")

TYPHOON_KEY : set
OCR_MODEL   : typhoon-ocr
LLM_MODEL   : typhoon-v2.5-30b-a3b-instruct


In [4]:
# ============================================================
# CELL 3: Helpers — rate limiter, paths, metadata
# ============================================================

class RateLimiter:
    def __init__(self, max_calls, period=60):
        self.max_calls = max_calls
        self.period    = period
        self.calls     = deque()
        self.lock      = threading.Lock()

    def wait(self):
        with self.lock:
            now = time.time()
            while self.calls and now - self.calls[0] >= self.period:
                self.calls.popleft()
            if len(self.calls) >= self.max_calls:
                sleep_time = self.period - (now - self.calls[0]) + 0.05
                time.sleep(max(0, sleep_time))
                now = time.time()
                while self.calls and now - self.calls[0] >= self.period:
                    self.calls.popleft()
            self.calls.append(time.time())

ocr_limiter = RateLimiter(OCR_LIMIT_PER_MIN, 60)
llm_limiter = RateLimiter(LLM_LIMIT_PER_MIN, 60)

def format_seconds(s): return f"{s:.2f}s"

def thai_to_int(value):
    if value is None: return None
    s = str(value).translate(str.maketrans("๐๑๒๓๔๕๖๗๘๙","0123456789")).strip()
    s = s.replace(",","").replace(" ","")
    m = re.search(r"-?\d+", s)
    return int(m.group(0)) if m else None

def detect_form_type(pdf_path):
    return "party_list" if "บช" in Path(pdf_path).name else "constituency"

def extract_path_metadata(pdf_path):
    parts = Path(pdf_path).parts
    try:
        amphoe   = parts[-4]
        tambon   = parts[-3]
        m        = re.search(r"(\d+)$", parts[-2])
        unit_num = int(m.group(1)) if m else 0
    except Exception:
        amphoe, tambon, unit_num = "unknown","unknown",0
    return amphoe, tambon, unit_num

def llm_json_path(pdf_path):
    pdf = Path(pdf_path)
    try:    rel = pdf.relative_to(Path(RAW_DATA_DIR))
    except: rel = Path(pdf.name)
    out = Path(OUTPUT_DIR) / rel.with_suffix(".json")
    out.parent.mkdir(parents=True, exist_ok=True)
    return out

def ocr_txt_path(pdf_path):
    pdf = Path(pdf_path)
    try:    rel = pdf.relative_to(Path(RAW_DATA_DIR))
    except: rel = Path(pdf.name)
    out = Path(OCR_TEXT_DIR) / rel.parent / (pdf.stem + "_ocr.txt")
    out.parent.mkdir(parents=True, exist_ok=True)
    return out

def get_all_pdfs():
    all_pdfs = sorted(Path(RAW_DATA_DIR).rglob("*.pdf"))
    print(f"Found: {len(all_pdfs)} PDFs")
    return all_pdfs

def write_timing_reports():
    def _write_csv(path, rows):
        if not rows: return
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fieldnames = sorted({k for row in rows for k in row.keys()})
        with open(path,"w",newline="",encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(rows)
        print(f"Timing → {path} ({len(rows)} rows)")
    _write_csv(Path(OUTPUT_DIR)/"ocr_request_timings.csv", OCR_REQUEST_TIMINGS)
    _write_csv(Path(OUTPUT_DIR)/"ocr_file_timings.csv",    OCR_FILE_TIMINGS)
    _write_csv(Path(OUTPUT_DIR)/"llm_request_timings.csv", LLM_REQUEST_TIMINGS)
    _write_csv(Path(OUTPUT_DIR)/"llm_file_timings.csv",    LLM_FILE_TIMINGS)

print("Helpers loaded")

Helpers loaded


In [5]:
# ============================================================
# CELL 4: OCR functions — direct Typhoon API with custom prompt
# ============================================================

TYPHOON_OCR_PROMPT = """Below is a scanned Thai government election result form.
Extract ALL text and numbers from the image exactly as they appear.

This form has two types:
- ส.ส.5/18 (แบบแบ่งเขตเลือกตั้ง) = constituency form
- ส.ส.5/18 (บช) (แบบบัญชีรายชื่อ) = party-list form

Extract in this order:

1. HEADER: form type from bracket, province, district, constituency number, polling unit number.

2. SUMMARY SECTION — read each number carefully:
   ๑.๑ = eligible_voters
   ๑.๒ = turnout
   ๒.๑ = ballots_allocated
   ๒.๒ = ballots_used
   ๒.๒.๑ บัตรดี = valid_ballots
   ๒.๒.๒ บัตรเสีย = spoiled_ballots
   ๒.๒.๓ บัตรที่ไม่เลือก = abstain_ballots
   ๒.๓ = ballots_remaining

3. RESULTS TABLE — extract every row:
   NUMBER | NAME (constituency) | PARTY | VOTES_NUMBER (VOTES_THAI_WORD)

Output clean structured text. Preserve all numbers. Do not skip any row."""


def pdf_to_images(pdf_path, dpi=300):
    tmp = None
    try:
        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as f:
            tmp = f.name
        shutil.copy2(pdf_path, tmp)
        doc    = fitz.open(tmp)
        pages  = []
        zoom   = dpi / 72
        matrix = fitz.Matrix(zoom, zoom)
        for page in doc:
            pix = page.get_pixmap(matrix=matrix)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            pages.append(img)
        doc.close()
        return pages
    finally:
        if tmp and os.path.exists(tmp): os.unlink(tmp)

def save_image_tmp(page_img, index):
    tmp_dir = Path("_tmp_imgs")
    tmp_dir.mkdir(exist_ok=True)
    img = page_img.convert("RGB")
    if max(img.size) > MAX_IMAGE_SIDE:
        img.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
    path = tmp_dir / f"page_{index:03d}.jpg"
    quality = 90
    while True:
        img.save(str(path), "JPEG", quality=quality, optimize=True)
        if path.stat().st_size <= MAX_IMAGE_BYTES or quality <= 55:
            break
        quality -= 10
    return str(path)

def pil_to_b64(img_path):
    with open(img_path, "rb") as f:
        return base64.b64encode(f.read()).decode()

def ocr_pdf_to_text(pdf_path, force=False):
    txt_path = ocr_txt_path(pdf_path)
    if txt_path.exists() and txt_path.stat().st_size > 50 and not force:
        print(f"    ↩️  OCR cached: {txt_path.name}")
        return txt_path.read_text(encoding="utf-8")

    convert_start = time.perf_counter()
    pages         = pdf_to_images(pdf_path)
    print(f"    🖼️  PDF→image: {format_seconds(time.perf_counter()-convert_start)} ({len(pages)} pages)")

    all_text = []

    for i, page in enumerate(pages):
        img_path  = save_image_tmp(page, i)
        img_bytes = Path(img_path).stat().st_size
        print(f"    🔍 OCR page {i+1}/{len(pages)} ({img_bytes/1_000_000:.2f} MB)...", end=" ", flush=True)

        page_result = {}
        page_error  = {}

        def _ocr(_img=img_path):
            request_start = time.perf_counter()
            wait_start    = time.perf_counter()
            try:
                ocr_limiter.wait()
                wait_seconds = time.perf_counter() - wait_start
                api_start    = time.perf_counter()
                b64          = pil_to_b64(_img)
                response     = typhoon_client.chat.completions.create(
                    model       = OCR_MODEL,
                    max_tokens  = 4096,
                    temperature = 0.0,
                    messages    = [{"role": "user", "content": [
                        {"type": "image_url",
                         "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                        {"type": "text", "text": TYPHOON_OCR_PROMPT}
                    ]}]
                )
                md          = response.choices[0].message.content
                api_seconds = time.perf_counter() - api_start
                total_s     = time.perf_counter() - request_start
                page_result["text"]   = md
                page_result["timing"] = {
                    "file": str(pdf_path), "page": i+1, "status": "PASS",
                    "chars": len(md), "image_mb": round(img_bytes/1_000_000,3),
                    "wait_seconds": round(wait_seconds,3),
                    "api_seconds":  round(api_seconds,3),
                    "total_seconds":round(total_s,3),
                }
            except Exception as e:
                total_s = time.perf_counter() - request_start
                page_error["err"] = str(e)
                page_result["timing"] = {
                    "file": str(pdf_path), "page": i+1, "status": "FAIL",
                    "chars": 0, "image_mb": round(img_bytes/1_000_000,3),
                    "wait_seconds": None, "api_seconds": None,
                    "total_seconds": round(total_s,3), "error": str(e)[:200],
                }

        pt = threading.Thread(target=_ocr, daemon=True)
        pt.start()
        pt.join(timeout=OCR_PAGE_TIMEOUT)

        timing = page_result.get("timing")
        if pt.is_alive():
            timing = {
                "file": str(pdf_path), "page": i+1, "status": "TIMEOUT",
                "chars": 0, "image_mb": round(img_bytes/1_000_000,3),
                "wait_seconds": None, "api_seconds": None,
                "total_seconds": OCR_PAGE_TIMEOUT,
                "error": f"timeout after {OCR_PAGE_TIMEOUT}s",
            }
            OCR_REQUEST_TIMINGS.append(timing)
            print(f"⏱️ timeout → skipped")
            all_text.append("")
            continue

        if timing: OCR_REQUEST_TIMINGS.append(timing)

        if "err" in page_error:
            print(f"❌ {page_error['err'][:100]}")
            all_text.append("")
            continue

        md = page_result.get("text","")
        all_text.append(md)
        print(f"✓ chars={len(md)} wait={format_seconds(timing['wait_seconds'])} api={format_seconds(timing['api_seconds'])} total={format_seconds(timing['total_seconds'])}")

    full_text = "\n\n--- PAGE BREAK ---\n\n".join(all_text)
    if full_text.strip():
        txt_path.write_text(full_text, encoding="utf-8")
        print(f"    💾 Saved → {txt_path}")
    else:
        print("    ⚠️ All pages empty — not cached")
    return full_text


def run_ocr_stage(pdf_paths, force=False):
    ocr_failed  = []
    stage_start = time.perf_counter()
    total       = len(pdf_paths)

    for i, pdf in enumerate(pdf_paths):
        file_start = time.perf_counter()
        txt = ocr_txt_path(str(pdf))
        if txt.exists() and txt.stat().st_size > 50 and not force:
            elapsed = time.perf_counter() - file_start
            print(f"[{i+1}/{total}] OCR SKIP: {pdf.name} | total={format_seconds(elapsed)}")
            OCR_FILE_TIMINGS.append({"file":str(pdf),"status":"SKIP","pages":None,"chars":txt.stat().st_size,"total_seconds":round(elapsed,3)})
            continue

        print(f"[{i+1}/{total}] OCR: {pdf}")
        try:
            text = ocr_pdf_to_text(str(pdf), force=force)
            if not text.strip(): raise ValueError("Empty OCR output")
            elapsed    = time.perf_counter() - file_start
            page_count = text.count("--- PAGE BREAK ---") + 1
            OCR_FILE_TIMINGS.append({"file":str(pdf),"status":"PASS","pages":page_count,"chars":len(text),"total_seconds":round(elapsed,3)})
            print(f"  ✓ OCR DONE: pages={page_count} chars={len(text)} total={format_seconds(elapsed)}")
        except Exception as e:
            elapsed = time.perf_counter() - file_start
            print(f"  ✗ OCR FAILED: {e} | total={format_seconds(elapsed)}")
            ocr_failed.append({"file":str(pdf),"error":str(e)})
            OCR_FILE_TIMINGS.append({"file":str(pdf),"status":"FAIL","pages":None,"chars":0,"total_seconds":round(elapsed,3),"error":str(e)[:200]})

    if ocr_failed:
        Path(OUTPUT_DIR,"_ocr_failed.json").write_text(json.dumps(ocr_failed,ensure_ascii=False,indent=2))
        print(f"OCR failed: {len(ocr_failed)}")

    stage_elapsed = time.perf_counter() - stage_start
    print(f"OCR STAGE DONE: files={total} failed={len(ocr_failed)} total={format_seconds(stage_elapsed)}")
    write_timing_reports()
    return ocr_failed

print("OCR functions loaded")

OCR functions loaded


In [9]:
# ============================================================
# CELL 5: LLM functions — friend's exact pipeline
# ============================================================
import difflib

VALID_PARTIES_LIST = [p.strip() for p in VALID_PARTIES_TEXT.replace("\n","").split(",") if p.strip()]

def normalize_party(party_raw: str) -> tuple:
    """Returns (matched_name, score, status)"""
    if not party_raw or not party_raw.strip():
        return party_raw, 0.0, "empty"
    p = party_raw.strip()
    if p in VALID_PARTIES_LIST:
        return p, 1.0, "exact_match"
    matches = difflib.get_close_matches(p, VALID_PARTIES_LIST, n=1, cutoff=0.6)
    if matches:
        score = difflib.SequenceMatcher(None, p, matches[0]).ratio()
        return matches[0], round(score, 4), "auto_matched"
    return p, 0.0, "unmatched"

def parse_thai_number(text: str):
    """Convert Thai number word to int. Returns None if unparseable."""
    THAI_ONES  = {"ศูนย์":0,"หนึ่ง":1,"สอง":2,"สาม":3,"สี่":4,"ห้า":5,"หก":6,"เจ็ด":7,"แปด":8,"เก้า":9}
    THAI_TENS  = {"สิบ":10,"ยี่สิบ":20,"สามสิบ":30,"สี่สิบ":40,"ห้าสิบ":50,"หกสิบ":60,"เจ็ดสิบ":70,"แปดสิบ":80,"เก้าสิบ":90}
    THAI_HUNDS = {"ร้อย":100,"สองร้อย":200,"สามร้อย":300,"สี่ร้อย":400,"ห้าร้อย":500,"หกร้อย":600,"เจ็ดร้อย":700,"แปดร้อย":800,"เก้าร้อย":900}
    THAI_THOUS = {"พัน":1000,"สองพัน":2000,"สามพัน":3000,"สี่พัน":4000,"ห้าพัน":5000}

    if not text or not text.strip(): return None
    text = text.strip()

    # Try direct lookup
    all_map = {**THAI_ONES, **THAI_TENS, **THAI_HUNDS, **THAI_THOUS}
    if text in all_map: return all_map[text]

    # Try compound: e.g. "หนึ่งร้อยสี่สิบหก"
    result = 0
    remaining = text
    for word, val in sorted(THAI_THOUS.items(), key=lambda x: -x[1]):
        if remaining.startswith(word): result += val; remaining = remaining[len(word):]
    for word, val in sorted(THAI_HUNDS.items(), key=lambda x: -x[1]):
        if remaining.startswith(word): result += val; remaining = remaining[len(word):]
    for word, val in sorted(THAI_TENS.items(), key=lambda x: -x[1]):
        if remaining.startswith(word): result += val; remaining = remaining[len(word):]
    for word, val in sorted(THAI_ONES.items(), key=lambda x: -x[1]):
        if remaining == word: result += val; remaining = ""

    if not remaining and result >= 0: return result
    return None

def enrich_results(results: list) -> list:
    """Add party_raw, party_match_score, votes_th_value, vote_confidence, vote_flags."""
    enriched = []
    for r in results:
        party_raw             = r.get("party","")
        party_matched, score, status = normalize_party(party_raw)
        votes                 = r.get("votes", 0)
        votes_th              = r.get("votes_th","")
        votes_th_value        = parse_thai_number(votes_th)
        vote_flags            = []

        # Confidence
        if votes_th_value is not None:
            if votes_th_value == votes:
                vote_confidence = "high"
            else:
                vote_confidence = "low"
                vote_flags.append("numeric_thai_text_mismatch")
        elif votes_th == "" or votes_th is None:
            vote_confidence = "medium"
        else:
            vote_confidence = "medium"

        enriched.append({
            "number":             r.get("number"),
            "name":               r.get("name",""),
            "party_raw":          party_raw,
            "party":              party_matched,
            "party_match_score":  score,
            "party_match_status": status,
            "votes":              votes,
            "votes_th":           votes_th,
            "votes_th_value":     votes_th_value,
            "vote_confidence":    vote_confidence,
            "vote_flags":         vote_flags,
        })
    return enriched

def build_rich_validation(results: list, summary: dict, retry_count: int = 0) -> dict:
    """Build rich _validation block matching friend's format."""
    vote_sum      = sum(r.get("votes", 0) for r in results)
    valid         = summary.get("valid_ballots") or 0
    used          = summary.get("ballots_used") or 0
    spoiled       = summary.get("spoiled_ballots") or 0
    abstain       = summary.get("abstain_ballots") or 0
    allocated     = summary.get("ballots_allocated") or 0
    remaining     = summary.get("ballots_remaining") or 0
    turnout       = summary.get("turnout") or 0

    vote_diff       = vote_sum - valid
    error_ratio     = abs(vote_diff) / valid if valid else 0.0
    ballot_bal_diff = (valid + spoiled + abstain) - used

    errors   = []
    warnings = []
    suggestions = []
    suspicious  = []

    if vote_diff != 0:
        if error_ratio > 0.05:
            errors.append(f"vote sum mismatch: results={vote_sum}, valid_ballots={valid}")
        else:
            warnings.append(f"small vote mismatch: effective_sum={vote_sum}, valid_ballots={valid}")
    if ballot_bal_diff != 0:
        errors.append(f"ballot used mismatch: valid+spoiled+abstain={valid+spoiled+abstain}, ballots_used={used}")
    if used > allocated and allocated > 0:
        errors.append(f"ballots_used > ballots_allocated: {used}>{allocated}")
    if used + remaining != allocated and allocated > 0:
        warnings.append(f"allocated mismatch: used+remaining={used+remaining}, allocated={allocated}")
    if turnout != used and turnout > 0 and used > 0:
        warnings.append(f"turnout and ballots_used differ: turnout={turnout}, ballots_used={used}")

    # Per-row checks
    low_conf = vlow_conf = 0
    for r in results:
        conf  = r.get("vote_confidence","high")
        flags = r.get("vote_flags",[])
        if conf == "medium": low_conf  += 1
        if conf == "low":    vlow_conf += 1
        if "numeric_thai_text_mismatch" in flags:
            th_val = r.get("votes_th_value")
            warnings.append(f"numeric/Thai vote mismatch for number {r['number']}: numeric={r['votes']}, thai={th_val}, text={r.get('votes_th','')}")
            suspicious.append({
                "number": r["number"], "party": r["party"], "party_raw": r.get("party_raw",""),
                "votes": r["votes"], "votes_th": r.get("votes_th",""),
                "votes_th_value": th_val, "confidence": "low",
                "flags": flags, "reason": "low-confidence vote field"
            })
            if th_val is not None:
                suggestions.append({
                    "type": "thai_text_vote_correction",
                    "number": r["number"], "party": r["party"], "party_raw": r.get("party_raw",""),
                    "current_votes": r["votes"], "suggested_votes": th_val,
                    "confidence": "high",
                    "reason": f"Numeric vote {r['votes']} conflicts with Thai text value {th_val}."
                })

    unmatched  = sum(1 for r in results if r.get("party_match_status") == "unmatched")
    low_conf_p = sum(1 for r in results if r.get("party_match_score",1.0) < 0.8)
    auto_match = sum(1 for r in results if r.get("party_match_status") == "auto_matched")
    exact_match= sum(1 for r in results if r.get("party_match_status") == "exact_match")

    # Status + error_type
    if errors:
        if error_ratio > 0.1:
            status = "REVIEW"; error_type = "large_vote_mismatch"
        else:
            status = "REVIEW"; error_type = "vote_mismatch"
    elif warnings:
        status = "WARNING"; error_type = "small_vote_mismatch_warning"
    else:
        status = "PASS"; error_type = "clean"

    retry_recommended = status == "REVIEW" and retry_count == 0

    return {
        "status":                         status,
        "error_type":                     error_type,
        "total_votes_in_table":           vote_sum,
        "effective_vote_sum":             vote_sum,
        "valid_ballots":                  valid,
        "vote_diff":                      vote_diff,
        "error_ratio":                    round(error_ratio, 6),
        "ballot_balance_diff":            ballot_bal_diff,
        "excluded_vote_rows":             [],
        "low_confidence_vote_count":      low_conf,
        "very_low_confidence_vote_count": vlow_conf,
        "party_issue_count":              unmatched + low_conf_p,
        "unmatched_party_count":          unmatched,
        "low_confidence_party_count":     low_conf_p,
        "auto_matched_party_count":       auto_match,
        "exact_party_count":              exact_match,
        "errors":                         errors,
        "warnings":                       warnings,
        "suggestions":                    suggestions,
        "suspicious_rows":                suspicious,
        "retry_recommended":              retry_recommended,
        "retry_count":                    retry_count,
        "used_fast_parser":               False,
        "used_llm":                       True,
    }


def compact_ocr_text(ocr_text, form_type):
    pages = re.split(r"\n\s*--- PAGE BREAK ---\s*\n", ocr_text)
    kept  = []
    for page in pages:
        if "ลงชื่อ" in page and "<table" not in page and "จำนวนบัตรเลือกตั้ง" not in page:
            continue
        if any(k in page for k in ["จำนวนผู้มีสิทธิ","จำนวนบัตรเลือกตั้ง","<table","รวมคะแนนทั้งสิ้น"]):
            kept.append(page)
    text = "\n\n--- PAGE BREAK ---\n\n".join(kept) if kept else ocr_text
    return text[:9000]


def build_llm_prompt(ocr_text, metadata, validation_feedback=None, previous_json=None):
    form_type    = metadata.get("form_type") or detect_form_type(metadata.get("source",""))
    compact_text = compact_ocr_text(ocr_text, form_type)
    result_label = (
        "candidate rows from the constituency table; include candidate name if visible"
        if form_type == "constituency"
        else "party-list rows from all party-list pages"
    )
    feedback = ""
    if validation_feedback:
        feedback = f"""
Previous extraction failed validation:
{json.dumps(validation_feedback, ensure_ascii=False)}
Fix the extracted values using the OCR text. Prefer numeric digits over Thai words if they conflict.
"""
    prev = ""
    if previous_json:
        prev = f"""
Previous JSON:
{json.dumps(previous_json, ensure_ascii=False)}
"""
    return f"""You are a Thai election data extraction system.
Return ONLY valid JSON. Do not include markdown. Do not explain.

Form type hint: {form_type}
Valid party names for correction:
{VALID_PARTIES_TEXT}

Required JSON structure:
{{
  "form_type": "{form_type}",
  "summary": {{
    "eligible_voters": int_or_null,
    "turnout": int_or_null,
    "ballots_allocated": int_or_null,
    "ballots_used": int_or_null,
    "valid_ballots": int_or_null,
    "spoiled_ballots": int_or_null,
    "abstain_ballots": int_or_null,
    "ballots_remaining": int_or_null
  }},
  "results": [
    {{
      "number": int,
      "name": string_or_empty,
      "party": string,
      "votes": int,
      "votes_th": string_or_empty
    }}
  ]
}}

Rules:
- Extract {result_label}.
- Convert Thai digits to Arabic integers.
- Use null only if a summary value is truly missing.
- Use 0 for actual zero votes.
- IMPORTANT: valid_ballots must come from the summary field "บัตรดี", NOT from the table total row.
- The table total row may be wrong. Extract row votes as seen; do not change votes to force consistency.
- Prefer numeric digits over Thai words when OCR conflicts.
- For constituency, party is the candidate's political party.
- For party_list, party is the party name and name should be empty.
- Do not invent party names. Extract the party text as seen in OCR.
{feedback}
{prev}
OCR TEXT:
{compact_text}
"""


def load_json_from_llm_text(raw):
    raw   = raw.strip()
    raw   = re.sub(r"^```json\s*","",raw)
    raw   = re.sub(r"^```\s*","",raw)
    raw   = re.sub(r"\s*```$","",raw).strip()
    start = raw.find("{")
    end   = raw.rfind("}")
    if start != -1 and end != -1 and end > start:
        raw = raw[start:end+1]
    return json.loads(raw)


def call_llm(prompt, request_label="llm", source=None):
    headers = {"Authorization": f"Bearer {TYPHOON_KEY}", "Content-Type": "application/json"}
    body    = {"model": LLM_MODEL, "messages": [{"role":"user","content":prompt}], "max_tokens": 4096, "temperature": 0.0}

    for attempt in range(3):
        request_start = time.perf_counter()
        wait_start    = time.perf_counter()
        llm_limiter.wait()
        wait_seconds  = time.perf_counter() - wait_start
        api_start     = time.perf_counter()
        try:
            resp        = req.post(f"{TYPHOON_BASE_URL}/chat/completions", headers=headers, json=body, timeout=90)
            api_seconds = time.perf_counter() - api_start
            total_s     = time.perf_counter() - request_start
            timing_row  = {
                "file": source, "request_label": request_label, "attempt": attempt+1,
                "status_code": resp.status_code, "prompt_chars": len(prompt),
                "wait_seconds": round(wait_seconds,3), "api_seconds": round(api_seconds,3),
                "total_seconds": round(total_s,3),
            }
            if resp.status_code == 429:
                timing_row["status"] = "RATE_LIMIT"
                LLM_REQUEST_TIMINGS.append(timing_row)
                print(f"    ⏳ LLM rate-limited attempt={attempt+1} wait={format_seconds(wait_seconds)} total={format_seconds(total_s)}")
                time.sleep(20*(attempt+1))
                continue
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"]
            timing_row["status"]         = "PASS"
            timing_row["response_chars"] = len(content)
            LLM_REQUEST_TIMINGS.append(timing_row)
            print(f"    🤖 LLM {request_label} attempt={attempt+1} prompt={len(prompt)} resp={len(content)} wait={format_seconds(wait_seconds)} api={format_seconds(api_seconds)} total={format_seconds(total_s)}")
            return content
        except Exception as e:
            api_seconds = time.perf_counter() - api_start
            total_s     = time.perf_counter() - request_start
            LLM_REQUEST_TIMINGS.append({
                "file": source, "request_label": request_label, "attempt": attempt+1,
                "status": "FAIL", "prompt_chars": len(prompt),
                "wait_seconds": round(wait_seconds,3), "api_seconds": round(api_seconds,3),
                "total_seconds": round(total_s,3), "error": str(e)[:200],
            })
            if attempt == 2: raise
            time.sleep(5*(attempt+1))
    raise RuntimeError("LLM retry exhausted")


def normalize_record(data, metadata):
    form_type = data.get("form_type") or metadata.get("form_type") or detect_form_type(metadata.get("source",""))
    data["form_type"] = form_type if form_type in ["constituency","party_list"] else metadata.get("form_type")
    summary = data.get("summary") or {}
    for k in ["eligible_voters","turnout","ballots_allocated","ballots_used","valid_ballots","spoiled_ballots","abstain_ballots","ballots_remaining"]:
        summary[k] = thai_to_int(summary.get(k))
    data["summary"] = summary
    clean = []
    for r in data.get("results",[]):
        number = thai_to_int(r.get("number"))
        votes  = thai_to_int(r.get("votes"))
        if number is None: continue
        clean.append({
            "number":   number,
            "name":     str(r.get("name","") or "").strip(),
            "party":    str(r.get("party","") or "").strip(),
            "votes":    votes if votes is not None else 0,
            "votes_th": str(r.get("votes_th","") or "").strip(),
        })
    data["results"] = sorted(clean, key=lambda x: x.get("number",0))
    return data


def validate_record(data):
    errors   = []
    warnings = []
    summary  = data.get("summary",{})
    results  = data.get("results",[])
    form_type= data.get("form_type")

    if form_type not in ["constituency","party_list"]:
        errors.append("invalid form_type")
    if not results:
        errors.append("empty results")

    nums = [r.get("number") for r in results]
    if len(nums) != len(set(nums)):
        errors.append("duplicate result number")

    for r in results:
        if r.get("number") is None:   errors.append("missing result number")
        if r.get("votes") is None:    errors.append(f"missing votes for number {r.get('number')}")
        elif r.get("votes") < 0:      errors.append(f"negative votes for number {r.get('number')}")
        if not r.get("party"):        warnings.append(f"missing party for number {r.get('number')}")

    vote_sum  = sum(r.get("votes",0) for r in results)
    valid     = summary.get("valid_ballots")
    used      = summary.get("ballots_used")
    spoiled   = summary.get("spoiled_ballots")
    abstain   = summary.get("abstain_ballots")
    allocated = summary.get("ballots_allocated")
    remaining = summary.get("ballots_remaining")
    turnout   = summary.get("turnout")

    if valid is not None and vote_sum != valid:
        errors.append(f"vote sum mismatch: results={vote_sum}, valid_ballots={valid}")
    if None not in [valid,spoiled,abstain,used] and valid+spoiled+abstain != used:
        errors.append(f"ballot used mismatch: valid+spoiled+abstain={valid+spoiled+abstain}, ballots_used={used}")
    if None not in [used,allocated] and used > allocated:
        errors.append(f"ballots_used > ballots_allocated: {used}>{allocated}")
    if None not in [used,remaining,allocated] and used+remaining != allocated:
        warnings.append(f"allocated mismatch: used+remaining={used+remaining}, allocated={allocated}")
    if None not in [turnout,used] and turnout != used:
        warnings.append(f"turnout and ballots_used differ: turnout={turnout}, ballots_used={used}")

    status = "REVIEW" if errors else "WARNING" if warnings else "PASS"
    return {"total_votes_in_table": vote_sum, "status": status, "errors": errors, "warnings": warnings}


def llm_parse_to_json(ocr_text, metadata, validation_feedback=None, previous_json=None):
    form_type             = metadata.get("form_type") or detect_form_type(metadata.get("source",""))
    metadata              = dict(metadata)
    metadata["form_type"] = form_type
    prompt                = build_llm_prompt(ocr_text, metadata, validation_feedback, previous_json)
    request_label         = "retry" if validation_feedback else "initial"
    raw                   = call_llm(prompt, request_label=request_label, source=metadata.get("source"))
    data                  = load_json_from_llm_text(raw)
    data                  = normalize_record(data, metadata)
    enriched              = enrich_results(data.get("results", []))
    data["results"]       = enriched
    validation            = build_rich_validation(enriched, data.get("summary", {}))
    return {
        "metadata":    metadata,
        "form_type":   data.get("form_type", form_type),
        "summary":     data.get("summary", {}),
        "results":     enriched,
        "_validation": validation,
    }


def llm_parse_with_retry(ocr_text, metadata, max_retries=1):
    result      = llm_parse_to_json(ocr_text, metadata)
    retry_count = 0
    for _ in range(max_retries):
        if result["_validation"]["status"] in ["PASS", "WARNING"]:
            break
        retry_count += 1
        print(f"    🔁 Retry {retry_count}: {result['_validation'].get('errors',[])}")
        result = llm_parse_to_json(
            ocr_text, metadata,
            validation_feedback = result["_validation"],
            previous_json       = {"form_type": result.get("form_type"), "summary": result.get("summary"), "results": result.get("results")},
        )
    result["_validation"]["retry_count"] = retry_count
    result["_validation"]["retry_recommended"] = (
        result["_validation"]["status"] == "REVIEW" and retry_count >= max_retries
    )
    return result



def result_rows_from_json(result, pdf_path):
    amphoe, tambon, unit_num = extract_path_metadata(pdf_path)
    rows     = []
    summary  = result.get("summary",{})
    validation = result.get("_validation",{})
    for r in result.get("results",[]):
        rows.append({
            "province": PROVINCE, "constituency": CONSTITUENCY,
            "amphoe": amphoe, "tambon": tambon, "unit": unit_num,
            "form_type":          result.get("form_type", detect_form_type(pdf_path)),
            "eligible_voters":    summary.get("eligible_voters"),
            "turnout":            summary.get("turnout"),
            "ballots_allocated":  summary.get("ballots_allocated"),
            "ballots_used":       summary.get("ballots_used"),
            "valid_ballots":      summary.get("valid_ballots"),
            "spoiled_ballots":    summary.get("spoiled_ballots"),
            "abstain_ballots":    summary.get("abstain_ballots"),
            "ballots_remaining":  summary.get("ballots_remaining"),
            "candidate_number":   r.get("number"),
            "candidate_name":     r.get("name",""),
            "party":              r.get("party",""),
            "votes":              r.get("votes",0),
            "votes_th":           r.get("votes_th",""),
            "validation":         validation.get("status",""),
            "validation_errors":  "; ".join(validation.get("errors",[])),
            "validation_warnings":"; ".join(validation.get("warnings",[])),
            "source":             str(pdf_path),
        })
    return rows


def validation_report_row(result, pdf_path):
    amphoe, tambon, unit_num = extract_path_metadata(pdf_path)
    v = result.get("_validation",{})
    return {
        "source":               str(pdf_path),
        "form_type":            result.get("form_type", detect_form_type(pdf_path)),
        "amphoe":               amphoe, "tambon": tambon, "unit": unit_num,
        "status":               v.get("status",""),
        "total_votes_in_table": v.get("total_votes_in_table",0),
        "valid_ballots":        result.get("summary",{}).get("valid_ballots"),
        "errors":               "; ".join(v.get("errors",[])),
        "warnings":             "; ".join(v.get("warnings",[])),
    }


def run_llm_stage(pdf_paths, force=False):
    all_records   = []
    reports       = []
    failed        = []
    manual_review = []
    stage_start   = time.perf_counter()
    total         = len(pdf_paths)

    for i, pdf in enumerate(pdf_paths):
        file_start  = time.perf_counter()
        out_json    = llm_json_path(str(pdf))
        txt_path    = ocr_txt_path(str(pdf))
        form_label  = detect_form_type(str(pdf))
        amphoe, tambon, unit_num = extract_path_metadata(str(pdf))

        if out_json.exists() and out_json.stat().st_size > 50 and not force:
            elapsed = time.perf_counter() - file_start
            print(f"[{i+1}/{total}] LLM SKIP: {pdf.name} | total={format_seconds(elapsed)}")
            try:
                result = json.loads(out_json.read_text(encoding="utf-8"))
                result["_validation"] = validate_record(result)
                all_records.extend(result_rows_from_json(result, str(pdf)))
                reports.append(validation_report_row(result, str(pdf)))
                if result["_validation"]["status"] == "REVIEW":
                    manual_review.append({"file":str(pdf),"ocr_text":str(txt_path),"json":str(out_json),"validation":result["_validation"]})
                LLM_FILE_TIMINGS.append({"file":str(pdf),"status":"SKIP","rows":len(result.get("results",[])),"total_seconds":round(elapsed,3)})
            except Exception as e:
                failed.append({"file":str(pdf),"stage":"load_existing_json","error":str(e)})
                LLM_FILE_TIMINGS.append({"file":str(pdf),"status":"FAIL_LOAD_CACHE","rows":0,"total_seconds":round(elapsed,3),"error":str(e)[:200]})
            continue

        print(f"[{i+1}/{total}] LLM: {amphoe}/{tambon}/หน่วย{unit_num} — {pdf.name}")
        try:
            if not txt_path.exists() or txt_path.stat().st_size <= 50:
                raise ValueError(f"Missing OCR text: {txt_path}")
            ocr_text = txt_path.read_text(encoding="utf-8")
            metadata = {
                "province": PROVINCE, "constituency": CONSTITUENCY,
                "amphoe": amphoe, "tambon": tambon, "unit": unit_num,
                "form_type": form_label, "source": str(pdf),
                "ocr_text": str(txt_path),
                "ocr_engine": f"{OCR_MODEL}+custom_prompt",
                "llm_parser": LLM_MODEL,
            }
            result = llm_parse_with_retry(ocr_text, metadata, max_retries=1)
            out_json.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
            all_records.extend(result_rows_from_json(result, str(pdf)))
            reports.append(validation_report_row(result, str(pdf)))
            if result["_validation"]["status"] == "REVIEW":
                manual_review.append({"file":str(pdf),"ocr_text":str(txt_path),"json":str(out_json),"validation":result["_validation"]})
            v       = result["_validation"]
            elapsed = time.perf_counter() - file_start
            LLM_FILE_TIMINGS.append({"file":str(pdf),"status":v["status"],"rows":len(result.get("results",[])),"total_votes":v["total_votes_in_table"],"retry_count":v.get("retry_count",0),"total_seconds":round(elapsed,3)})
            print(f"  ✓ {v['status']} rows={len(result.get('results',[]))} votes={v['total_votes_in_table']} total={format_seconds(elapsed)}")
        except Exception as e:
            elapsed = time.perf_counter() - file_start
            print(f"  ✗ FAILED: {e} | total={format_seconds(elapsed)}")
            failed.append({"file":str(pdf),"stage":"llm","error":str(e)})
            LLM_FILE_TIMINGS.append({"file":str(pdf),"status":"FAIL","rows":0,"total_seconds":round(elapsed,3),"error":str(e)[:200]})

    if all_records:
        csv_path = Path(OUTPUT_DIR)/"all_results.csv"
        with open(csv_path,"w",newline="",encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=list(all_records[0].keys()))
            writer.writeheader(); writer.writerows(all_records)
        print(f"Master CSV → {csv_path} ({len(all_records)} rows)")

    if reports:
        rp = Path(OUTPUT_DIR)/"validation_report.csv"
        with open(rp,"w",newline="",encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=list(reports[0].keys()))
            writer.writeheader(); writer.writerows(reports)
        print(f"Validation report → {rp} ({len(reports)} files)")

    if failed:
        Path(OUTPUT_DIR,"_failed.json").write_text(json.dumps(failed,ensure_ascii=False,indent=2))
        print(f"Failed: {len(failed)}")

    review_path = Path(OUTPUT_DIR)/"manual_review.json"
    review_path.write_text(json.dumps(manual_review,ensure_ascii=False,indent=2))
    print(f"Manual review → {review_path} ({len(manual_review)} files)")

    stage_elapsed = time.perf_counter() - stage_start
    print(f"LLM STAGE DONE: files={total} failed={len(failed)} review={len(manual_review)} rows={len(all_records)} total={format_seconds(stage_elapsed)}")
    write_timing_reports()
    return all_records, reports, failed, manual_review

print("LLM functions loaded")

LLM functions loaded


In [7]:
# ============================================================
# CELL 6: Run OCR stage
# Set force=True to re-OCR even when cached
# ============================================================
all_pdfs   = get_all_pdfs()
ocr_failed = run_ocr_stage(all_pdfs, force=False)

Found: 566 PDFs
[1/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[2/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[3/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[4/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[5/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[6/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[7/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[8/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[9/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[10/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[11/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[12/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[13/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[14/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[15/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[16/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[17/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[18/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[19/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[20/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[21/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[22/566] OCR SKIP: 5ทับ18

c:\Users\sukon\miniconda3\envs\my_python\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (98286371 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  ✗ OCR FAILED: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\sukon\\AppData\\Local\\Temp\\tmpewtuh4dw.pdf' | total=18.16s
[374/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[375/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[376/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[377/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[378/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[379/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[380/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[381/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[382/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[383/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[384/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[385/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[386/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[387/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[388/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[389/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[390/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[391/566] OCR SKIP: 

c:\Users\sukon\miniconda3\envs\my_python\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (89957868 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
c:\Users\sukon\miniconda3\envs\my_python\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (174587869 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


  ✗ OCR FAILED: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\sukon\\AppData\\Local\\Temp\\tmpthlyj0_s.pdf' | total=14.66s
[404/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[405/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[406/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[407/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[408/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[409/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[410/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[411/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[412/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[413/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[414/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[415/566] OCR SKIP: 5ทับ18  (บช).pdf | total=0.00s
[416/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[417/566] OCR SKIP: 5ทับ 18( บช).pdf | total=0.00s
[418/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[419/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[420/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[421/566] OCR SK

...

    🖼️  PDF→image: 25.25s (2 pages)
    🔍 OCR page 1/2 (0.62 MB)... ✓ chars=2703 wait=0.00s api=18.47s total=18.47s
    🔍 OCR page 2/2 (0.50 MB)... ✓ chars=908 wait=0.00s api=14.02s total=14.02s
    💾 Saved → ocr_texts_v2\เขต 2 อ.เมืองฯ\เขตไร่น้อย\เขตไร่น้อย\หน่วยเลือกตั้งที่ 34\5ทับ18_ocr.txt
  ✓ OCR DONE: pages=2 chars=3633 total=59.47s
[533/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[534/566] OCR: rawData\เขต 2 อ.เมืองฯ\เขตไร่น้อย\เขตไร่น้อย\หน่วยเลือกตั้งที่ 35\5ทับ18.pdf


c:\Users\sukon\miniconda3\envs\my_python\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (107143052 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
c:\Users\sukon\miniconda3\envs\my_python\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (106414550 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


    🖼️  PDF→image: 24.44s (2 pages)
    🔍 OCR page 1/2 (0.63 MB)... ✓ chars=2611 wait=0.00s api=18.31s total=18.31s
    🔍 OCR page 2/2 (0.53 MB)... ✓ chars=939 wait=0.00s api=9.15s total=9.15s
    💾 Saved → ocr_texts_v2\เขต 2 อ.เมืองฯ\เขตไร่น้อย\เขตไร่น้อย\หน่วยเลือกตั้งที่ 35\5ทับ18_ocr.txt
  ✓ OCR DONE: pages=2 chars=3572 total=53.63s
[535/566] OCR: rawData\เขต 2 อ.เมืองฯ\เขตไร่น้อย\เขตไร่น้อย\หน่วยเลือกตั้งที่ 36\5ทับ18(บช).pdf


c:\Users\sukon\miniconda3\envs\my_python\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (125669986 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
c:\Users\sukon\miniconda3\envs\my_python\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (101449098 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


    🖼️  PDF→image: 27.98s (2 pages)
    🔍 OCR page 1/2 (0.72 MB)... ✓ chars=2708 wait=0.00s api=18.21s total=18.21s
    🔍 OCR page 2/2 (0.57 MB)... ✓ chars=1019 wait=0.00s api=17.41s total=17.41s
    💾 Saved → ocr_texts_v2\เขต 2 อ.เมืองฯ\เขตไร่น้อย\เขตไร่น้อย\หน่วยเลือกตั้งที่ 36\5ทับ18(บช)_ocr.txt
  ✓ OCR DONE: pages=2 chars=3749 total=65.61s
[536/566] OCR: rawData\เขต 2 อ.เมืองฯ\เขตไร่น้อย\เขตไร่น้อย\หน่วยเลือกตั้งที่ 36\5ทับ18.pdf
    🖼️  PDF→image: 28.15s (2 pages)
    🔍 OCR page 1/2 (0.72 MB)... ✓ chars=2708 wait=0.00s api=15.44s total=15.44s
    🔍 OCR page 2/2 (0.57 MB)... ✓ chars=1019 wait=0.00s api=7.84s total=7.84s
    💾 Saved → ocr_texts_v2\เขต 2 อ.เมืองฯ\เขตไร่น้อย\เขตไร่น้อย\หน่วยเลือกตั้งที่ 36\5ทับ18_ocr.txt
  ✓ OCR DONE: pages=2 chars=3749 total=53.37s
[537/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[538/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[539/566] OCR SKIP: 5ทับ18(บช).pdf | total=0.00s
[540/566] OCR SKIP: 5ทับ18.pdf | total=0.00s
[541/566] OCR SKIP: 5ทับ18(บช).pdf

In [10]:
# ============================================================
# CELL 7: Run LLM stage
# Set force=True to re-parse even when JSON exists
# ============================================================
all_pdfs = get_all_pdfs()
all_records, reports, failed, manual_review = run_llm_stage(all_pdfs, force=False)

Found: 566 PDFs
[1/566] LLM: อำเภอเขื่องใน/ตำบลกลางใหญ่/หน่วย1 — 5ทับ18(บช).pdf
    🤖 LLM initial attempt=1 prompt=8640 resp=7191 wait=0.00s api=13.88s total=13.88s
  ✓ WARNING rows=57 votes=264 total=13.88s
[2/566] LLM: อำเภอเขื่องใน/ตำบลกลางใหญ่/หน่วย1 — 5ทับ18.pdf
    🤖 LLM initial attempt=1 prompt=5911 resp=1726 wait=0.00s api=5.29s total=5.29s
  ✓ WARNING rows=10 votes=253 total=5.30s
[3/566] LLM: อำเภอเขื่องใน/ตำบลกลางใหญ่/หน่วย10 — 5ทับ18(บช).pdf
    🤖 LLM initial attempt=1 prompt=8873 resp=7182 wait=0.00s api=14.05s total=14.05s
    🔁 Retry 1: ['vote sum mismatch: results=117, valid_ballots=127', 'ballot used mismatch: valid+spoiled+abstain=132, ballots_used=130']
    🤖 LLM retry attempt=1 prompt=25414 resp=7184 wait=0.00s api=15.19s total=15.19s
  ✓ REVIEW rows=57 votes=158 total=29.24s
[4/566] LLM: อำเภอเขื่องใน/ตำบลกลางใหญ่/หน่วย10 — 5ทับ18.pdf
    🤖 LLM initial attempt=1 prompt=4621 resp=1712 wait=0.00s api=3.74s total=3.74s
    🔁 Retry 1: ['vote sum mismatch: results=254, 

In [12]:
# ============================================================
# CELL 8: Write timing reports
# ============================================================
write_timing_reports()

Timing → output_v2\ocr_request_timings.csv (8 rows)
Timing → output_v2\ocr_file_timings.csv (566 rows)
Timing → output_v2\llm_request_timings.csv (1280 rows)
Timing → output_v2\llm_file_timings.csv (612 rows)


In [11]:
# ============================================================
# CELL 9: Re-run LLM for high-error files
# Finds JSON files where |total_votes - valid_ballots| > RERUN_THRESHOLD%
# Sends failed JSON + OCR text to LLM with correction prompt
# Picks the better result (fewer errors / more reasonable votes)
# ============================================================

RERUN_THRESHOLD = 20.0   # re-run files with error > this %

# ── Correction-specific prompt ────────────────────────────
def build_correction_prompt(ocr_text, metadata, failed_json):
    """
    Specialized prompt for correcting a previously failed extraction.
    Sends the failed JSON so the LLM can see what went wrong.
    Also instructs LLM to use votes_th to verify votes.
    """
    form_type    = metadata.get("form_type") or detect_form_type(metadata.get("source",""))
    compact_text = compact_ocr_text(ocr_text, form_type)

    # Compute what the issue was
    prev_results = failed_json.get("results", [])
    prev_sum     = sum(r.get("votes",0) for r in prev_results)
    prev_valid   = failed_json.get("summary",{}).get("valid_ballots") or 0
    prev_diff    = prev_sum - prev_valid

    # Find suspicious rows from failed JSON
    suspicious_rows = failed_json.get("_validation",{}).get("suspicious_rows",[])
    mismatch_rows   = [
        r for r in prev_results
        if r.get("votes_th_value") is not None
        and r.get("votes_th_value") != r.get("votes")
    ]

    mismatch_hint = ""
    if mismatch_rows:
        hints = []
        for r in mismatch_rows[:5]:   # top 5 mismatches
            hints.append(
                f"  Row #{r.get('number')} {r.get('party','')}: "
                f"votes={r.get('votes')} but votes_th_value={r.get('votes_th_value')} "
                f"(Thai word: {r.get('votes_th','')})"
            )
        mismatch_hint = "Rows where numeric votes conflict with Thai word value:\n" + "\n".join(hints)

    result_label = (
        "candidate rows from the constituency table"
        if form_type == "constituency"
        else "ALL 57 party rows from ALL pages"
    )

    return f"""You are a Thai election data correction system.
Return ONLY valid JSON. Do not include markdown. Do not explain.

TASK: The previous extraction had errors. Re-extract carefully to fix them.

PREVIOUS EXTRACTION HAD THIS PROBLEM:
- vote sum in table = {prev_sum}
- valid_ballots in summary = {prev_valid}
- difference = {prev_diff} ({abs(prev_diff/prev_valid*100) if prev_valid else 0:.1f}%)

PREVIOUS JSON (may have wrong values):
{json.dumps(failed_json, ensure_ascii=False)[:3000]}

{f"ROWS WITH NUMERIC vs THAI WORD MISMATCH (use Thai word to verify):\n{mismatch_hint}" if mismatch_hint else ""}

CORRECTION RULES:
1. Re-read EVERY vote number from the OCR text from scratch.
2. For each row, read BOTH the Arabic number AND the Thai word in brackets.
   - e.g. "146 ( หนึ่งร้อยสี่สิบหก )" → votes=146, votes_th="หนึ่งร้อยสี่สิบหก"
   - If Arabic number and Thai word disagree → trust the Thai word, it is handwritten clearly.
3. valid_ballots = ๒.๒.๑ บัตรดี from summary ONLY — never use table total.
4. Do NOT force votes to sum to valid_ballots. Extract as seen.
5. For party_list: extract ALL 57 parties, do not stop early.
6. Fix any party name typos against this list:
{VALID_PARTIES_TEXT}

Form type: {form_type}

Required JSON:
{{
  "form_type": "{form_type}",
  "summary": {{
    "eligible_voters": int_or_null,
    "turnout": int_or_null,
    "ballots_allocated": int_or_null,
    "ballots_used": int_or_null,
    "valid_ballots": int_or_null,
    "spoiled_ballots": int_or_null,
    "abstain_ballots": int_or_null,
    "ballots_remaining": int_or_null
  }},
  "results": [
    {{"number": int, "name": "string_or_empty", "party": "string", "votes": int, "votes_th": "string"}}
  ]
}}

OCR TEXT:
{compact_text}"""


def get_error_ratio_from_json(data: dict) -> float:
    """Get error ratio from a result JSON dict."""
    v        = data.get("_validation", {})
    if "error_ratio" in v:
        return float(v["error_ratio"])
    vote_sum = v.get("total_votes_in_table") or sum(r.get("votes",0) for r in data.get("results",[]))
    valid    = v.get("valid_ballots") or data.get("summary",{}).get("valid_ballots")
    if valid and valid > 0 and vote_sum is not None:
        return abs(vote_sum - valid) / valid
    return 0.0


def pick_better_result(original: dict, corrected: dict) -> tuple:
    """
    Compare two results and return the better one.
    Better = lower error ratio + more rows extracted.
    Returns (better_result, reason_string)
    """
    orig_err  = get_error_ratio_from_json(original)
    corr_err  = get_error_ratio_from_json(corrected)
    orig_rows = len(original.get("results", []))
    corr_rows = len(corrected.get("results", []))

    # If correction has significantly fewer rows → something went wrong
    if corr_rows < orig_rows * 0.7:
        return original, f"correction lost rows ({orig_rows}→{corr_rows}) → keeping original"

    # Lower error ratio wins
    if corr_err < orig_err:
        improvement = (orig_err - corr_err) * 100
        return corrected, f"correction better: {orig_err*100:.1f}%→{corr_err*100:.1f}% (improved {improvement:.1f}%)"
    elif orig_err <= corr_err:
        return original, f"original better or equal: {orig_err*100:.1f}% vs {corr_err*100:.1f}%"
    else:
        return corrected, f"correction chosen (rows: {corr_rows} vs {orig_rows})"


# ── Find all files needing re-run ─────────────────────────
all_jsons = sorted(Path(OUTPUT_DIR).rglob("*.json"))
all_jsons = [j for j in all_jsons if not j.name.startswith("_")]

rerun_files = []
for jf in all_jsons:
    try:
        data    = json.loads(jf.read_text(encoding="utf-8"))
        records = data if isinstance(data, list) else [data]
        for rec in records:
            if not isinstance(rec, dict): continue
            err = get_error_ratio_from_json(rec) * 100
            if err > RERUN_THRESHOLD:
                source = rec.get("metadata",{}).get("source","")
                rerun_files.append({
                    "json_path": str(jf),
                    "source":    source,
                    "error_pct": round(err, 1),
                    "record":    rec,
                })
    except: pass

print(f"Files needing re-run (error > {RERUN_THRESHOLD}%): {len(rerun_files)}")
print()

# ── Re-run LLM for each ───────────────────────────────────
improved  = 0
same      = 0
worse     = 0
rerun_log = []

for i, item in enumerate(rerun_files):
    pdf_path  = Path(item["source"])
    json_path = Path(item["json_path"])
    orig      = item["record"]
    err_pct   = item["error_pct"]

    print(f"[{i+1}/{len(rerun_files)}] {pdf_path.name} (error={err_pct}%)")

    # Check OCR text exists
    txt_path = ocr_txt_path(str(pdf_path))
    if not txt_path.exists():
        print(f"  ⚠️  No OCR text found at {txt_path} — skipping")
        rerun_log.append({"file": str(pdf_path), "status": "no_ocr", "original_error": err_pct})
        continue

    try:
        ocr_text = txt_path.read_text(encoding="utf-8")
        meta     = orig.get("metadata", {})

        # Build correction prompt with failed JSON
        prompt   = build_correction_prompt(ocr_text, meta, orig)

        print(f"  🤖 Sending correction prompt ({len(prompt)} chars)...")
        raw      = call_llm(prompt, request_label="correction", source=str(pdf_path))
        data     = load_json_from_llm_text(raw)
        data     = normalize_record(data, meta)
        enriched = enrich_results(data.get("results", []))
        data["results"] = enriched
        corrected = {
            "metadata":    meta,
            "form_type":   data.get("form_type", meta.get("form_type","")),
            "summary":     data.get("summary", {}),
            "results":     enriched,
            "_validation": build_rich_validation(enriched, data.get("summary",{})),
        }

        # Pick better result
        best, reason = pick_better_result(orig, corrected)
        new_err = get_error_ratio_from_json(best) * 100

        print(f"  → {reason}")
        print(f"  → error: {err_pct}% → {new_err:.1f}%")

        if new_err < err_pct - 1:
            improved += 1
            status = "improved"
        elif abs(new_err - err_pct) <= 1:
            same += 1
            status = "same"
        else:
            worse += 1
            status = "worse"

        # Save best result back to JSON
        json_path.write_text(
            json.dumps(best, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        rerun_log.append({
            "file":           str(pdf_path),
            "status":         status,
            "original_error": err_pct,
            "new_error":      round(new_err, 1),
            "reason":         reason,
        })

    except Exception as e:
        print(f"  ❌ {e}")
        rerun_log.append({
            "file":           str(pdf_path),
            "status":         "failed",
            "original_error": err_pct,
            "error":          str(e),
        })

# ── Save log ──────────────────────────────────────────────
log_path = Path(OUTPUT_DIR) / "_rerun_log.json"
log_path.write_text(json.dumps(rerun_log, ensure_ascii=False, indent=2))

print(f"\n{'='*50}")
print(f"Re-run complete")
print(f"  Improved : {improved}")
print(f"  Same     : {same}")
print(f"  Worse    : {worse}")
print(f"  Failed   : {sum(1 for r in rerun_log if r.get('status')=='failed')}")
print(f"  No OCR   : {sum(1 for r in rerun_log if r.get('status')=='no_ocr')}")
print(f"  Log → {log_path}")
print(f"\nRun Cell 8 (write_timing_reports) after this")

Files needing re-run (error > 20.0%): 262

[1/262] 5ทับ18(บช).pdf (error=24.4%)
  🤖 Sending correction prompt (11956 chars)...
    🤖 LLM correction attempt=1 prompt=11956 resp=4246 wait=0.00s api=9.41s total=9.41s
  → correction better: 24.4%→7.9% (improved 16.5%)
  → error: 24.4% → 7.9%
[2/262] 5ทับ18.pdf (error=96.9%)
  🤖 Sending correction prompt (7669 chars)...
    🤖 LLM correction attempt=1 prompt=7669 resp=1264 wait=0.00s api=4.21s total=4.21s
  → original better or equal: 96.9% vs 96.9%
  → error: 96.9% → 96.9%
[3/262] 5ทับ18(บช).pdf (error=102.0%)
  🤖 Sending correction prompt (11699 chars)...
    🤖 LLM correction attempt=1 prompt=11699 resp=4873 wait=0.00s api=11.80s total=11.80s
  → original better or equal: 102.0% vs 102.0%
  → error: 102.0% → 102.0%
[4/262] 5ทับ18.pdf (error=64.5%)
  🤖 Sending correction prompt (8870 chars)...
    🤖 LLM correction attempt=1 prompt=8870 resp=1280 wait=0.00s api=3.44s total=3.44s
  → original better or equal: 64.5% vs 64.5%
  → error: 64.5% →

In [ ]:
# ============================================================
# CELL 10: Re-OCR + Re-LLM for high-error files
# - Preprocesses images (contrast, sharpen) before OCR
# - Sends OCR errors in the prompt so model understands problem
# - Only runs on files with error > REOCR_THRESHOLD%
# ============================================================
from PIL import ImageEnhance, ImageFilter, ImageOps

REOCR_THRESHOLD  = 25.0   # only re-process files with error > this %
REOCR_DPI        = 400    # higher DPI for better quality
REOCR_TEXT_DIR   = "ocr_texts_rerun"   # save new OCR here
REOCR_OUTPUT_DIR = OUTPUT_DIR          # overwrite same JSON

Path(REOCR_TEXT_DIR).mkdir(exist_ok=True)


# ── Image preprocessing ───────────────────────────────────
def preprocess_for_ocr(img):
    """Enhance image quality before OCR."""
    img = img.convert("L")                          # grayscale
    img = ImageOps.autocontrast(img)                # auto levels
    img = ImageEnhance.Contrast(img).enhance(1.6)   # boost contrast
    img = ImageEnhance.Sharpness(img).enhance(1.3)  # sharpen
    return img.convert("RGB")


# ── OCR prompt that includes known errors ─────────────────
def build_ocr_prompt_with_errors(failed_json: dict) -> str:
    """Custom OCR prompt that tells the model what went wrong last time."""
    v         = failed_json.get("_validation", {})
    errors    = v.get("errors", [])
    vote_sum  = v.get("total_votes_in_table", 0)
    valid     = v.get("valid_ballots", 0)
    diff      = abs(vote_sum - valid)
    form_type = failed_json.get("form_type", "unknown")

    # Find rows with mismatch between votes and votes_th_value
    mismatch_hint = ""
    mismatch_rows = [
        r for r in failed_json.get("results", [])
        if r.get("votes_th_value") is not None
        and r.get("votes_th_value") != r.get("votes")
    ]
    if mismatch_rows:
        lines = []
        for r in mismatch_rows[:8]:
            lines.append(
                f"  - Row #{r.get('number')} {r.get('party','')}: "
                f"previously read as {r.get('votes')} but Thai word says {r.get('votes_th_value')} "
                f"(Thai: {r.get('votes_th','')})"
            )
        mismatch_hint = (
            "KNOWN ERRORS FROM PREVIOUS READ — please re-read these rows carefully:\n"
            + "\n".join(lines)
        )

    return f"""Below is a scanned Thai government election result form.
Extract ALL text and numbers from the image exactly as they appear.

IMPORTANT: A previous OCR attempt had errors:
- Vote sum read from table = {vote_sum}
- Valid ballots in summary (๒.๒.๑) = {valid}
- Difference = {diff} votes — this means {diff} votes were misread
{mismatch_hint}

Please re-read all numbers very carefully, especially:
1. Any number that looks similar to another (e.g. ๑/๗, ๓/๘, ๔/๙)
2. Numbers written with a flourish that could be misread
3. The vote count column (last column) for every row

This is a {'constituency (แบบแบ่งเขตเลือกตั้ง)' if form_type == 'constituency' else 'party-list (แบบบัญชีรายชื่อ)'} form.

SUMMARY SECTION — read each number carefully:
   ๑.๑ = eligible_voters, ๑.๒ = turnout
   ๒.๑ = ballots_allocated, ๒.๒ = ballots_used
   ๒.๒.๑ บัตรดี = valid_ballots (THIS IS THE KEY NUMBER)
   ๒.๒.๒ บัตรเสีย = spoiled_ballots
   ๒.๒.๓ บัตรที่ไม่เลือก = abstain_ballots
   ๒.๓ = ballots_remaining

RESULTS TABLE — extract EVERY row:
   NUMBER | NAME (if constituency) | PARTY | VOTES_NUMBER (VOTES_THAI_WORD)
   e.g. ๑ | นายวุฒิพงษ์ นามบุตร | กล้าธรรม | 146 (หนึ่งร้อยสี่สิบหก)

Output clean structured text. Do not skip any row. Do not guess — read exactly what is written."""


# ── Re-OCR one PDF with preprocessing ────────────────────
def reocr_pdf_with_preprocess(pdf_path: str, failed_json: dict, save_dir: str) -> str:
    """Re-OCR a PDF with image preprocessing + error-aware prompt."""
    pdf = Path(pdf_path)
    try:    rel = pdf.relative_to(RAW_DATA_DIR)
    except: rel = Path(pdf.name)
    txt_out = Path(save_dir) / rel.parent / (pdf.stem + "_ocr.txt")
    txt_out.parent.mkdir(parents=True, exist_ok=True)

    pages    = pdf_to_images(str(pdf), dpi=REOCR_DPI)
    all_text = []
    ocr_prompt = build_ocr_prompt_with_errors(failed_json)

    for i, page in enumerate(pages):
        print(f"    🔍 page {i+1}/{len(pages)}...", end=" ", flush=True)

        # Preprocess image
        processed = preprocess_for_ocr(page)

        # Save preprocessed image
        tmp_dir = Path("_tmp_reocr")
        tmp_dir.mkdir(exist_ok=True)
        img_path = str(tmp_dir / f"page_{i:03d}.jpg")
        processed.save(img_path, "JPEG", quality=95)

        page_result = {}
        page_error  = {}

        def _ocr(_p=img_path, _prompt=ocr_prompt):
            try:
                ocr_limiter.wait()
                with open(_p, "rb") as f:
                    b64 = base64.b64encode(f.read()).decode()
                resp = typhoon_client.chat.completions.create(
                    model=OCR_MODEL, max_tokens=4096, temperature=0.0,
                    messages=[{"role":"user","content":[
                        {"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{b64}"}},
                        {"type":"text","text":_prompt}
                    ]}]
                )
                page_result["text"] = resp.choices[0].message.content
            except Exception as e:
                page_error["err"] = str(e)

        t = threading.Thread(target=_ocr, daemon=True)
        t.start()
        t.join(timeout=OCR_PAGE_TIMEOUT)

        if t.is_alive():
            print(f"⏱️ timeout → skipped")
            all_text.append("")
            continue
        if "err" in page_error:
            print(f"❌ {page_error['err'][:60]}")
            all_text.append("")
            continue

        text = page_result.get("text","")
        all_text.append(text)
        print(f"✓ ({len(text)} chars)")

    full_text = "\n\n--- PAGE BREAK ---\n\n".join(all_text)
    if full_text.strip():
        txt_out.write_text(full_text, encoding="utf-8")
        print(f"    💾 New OCR → {txt_out}")
    return full_text


# ── Find high-error files ─────────────────────────────────
all_jsons    = sorted(Path(REOCR_OUTPUT_DIR).rglob("*.json"))
all_jsons    = [j for j in all_jsons if not j.name.startswith("_")]
reocr_files  = []

for jf in all_jsons:
    try:
        data    = json.loads(jf.read_text(encoding="utf-8"))
        records = data if isinstance(data, list) else [data]
        for rec in records:
            if not isinstance(rec, dict): continue
            v        = rec.get("_validation",{})
            err      = float(v.get("error_ratio", 0)) * 100
            if "error_ratio" not in v:
                vote_sum = v.get("total_votes_in_table",0) or sum(r.get("votes",0) for r in rec.get("results",[]))
                valid    = v.get("valid_ballots") or rec.get("summary",{}).get("valid_ballots") or 0
                err      = abs(vote_sum - valid) / valid * 100 if valid else 0
            if err > REOCR_THRESHOLD:
                source = rec.get("metadata",{}).get("source","")
                if source:
                    reocr_files.append({
                        "json_path": str(jf),
                        "source":    source,
                        "error_pct": round(err, 1),
                        "record":    rec,
                    })
    except: pass

print(f"Files to re-process (error > {REOCR_THRESHOLD}%): {len(reocr_files)}")
print(f"Each file: re-OCR at DPI={REOCR_DPI} with preprocessing + re-LLM")
print()

# ── Process each file ─────────────────────────────────────
improved  = 0
same      = 0
worse     = 0
log       = []

for i, item in enumerate(reocr_files):
    pdf_path  = item["source"]
    json_path = Path(item["json_path"])
    orig      = item["record"]
    err_pct   = item["error_pct"]

    print(f"[{i+1}/{len(reocr_files)}] {Path(pdf_path).name} (error={err_pct}%)")

    if not Path(pdf_path).exists():
        print(f"  ⚠️  PDF not found")
        log.append({"file": pdf_path, "status": "no_pdf", "original_error": err_pct})
        continue

    try:
        # ── Step 1: Re-OCR with preprocessing ────────────
        new_ocr = reocr_pdf_with_preprocess(pdf_path, orig, REOCR_TEXT_DIR)
        if not new_ocr.strip():
            print(f"  ⚠️  Re-OCR empty → skipping")
            log.append({"file": pdf_path, "status": "empty_ocr", "original_error": err_pct})
            continue

        # ── Step 2: Re-LLM with correction prompt ─────────
        meta     = dict(orig.get("metadata", {}))
        meta["ocr_engine"] = f"{OCR_MODEL}+preprocessed+custom_prompt"

        # First attempt
        result = llm_parse_to_json(new_ocr, meta)

        # Retry if still bad
        new_err = get_error_ratio_from_json(result) * 100
        if new_err > REOCR_THRESHOLD:
            print(f"  🔁 Still {new_err:.1f}% → retry with feedback")
            result = llm_parse_to_json(
                new_ocr, meta,
                validation_feedback = result["_validation"],
                previous_json       = {
                    "form_type": result.get("form_type"),
                    "summary":   result.get("summary"),
                    "results":   result.get("results"),
                },
            )

        new_err = get_error_ratio_from_json(result) * 100

        # ── Step 3: Pick better result ────────────────────
        orig_rows = len(orig.get("results", []))
        new_rows  = len(result.get("results", []))

        if new_rows < orig_rows * 0.7:
            # New result lost too many rows → keep original
            best   = orig
            reason = f"new result lost rows ({orig_rows}→{new_rows}) → keeping original"
            new_err = err_pct
        elif new_err < err_pct - 1:
            best   = result
            reason = f"improved {err_pct:.1f}%→{new_err:.1f}%"
        else:
            best   = orig
            reason = f"no improvement ({err_pct:.1f}%→{new_err:.1f}%) → keeping original"
            new_err = err_pct

        print(f"  → {reason}")

        # ── Step 4: Save best result ──────────────────────
        json_path.write_text(
            json.dumps(best, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        # Track stats
        if new_err < err_pct - 1:   improved += 1; status = "improved"
        elif new_err > err_pct + 1: worse    += 1; status = "worse"
        else:                       same     += 1; status = "same"

        log.append({
            "file":           pdf_path,
            "status":         status,
            "original_error": err_pct,
            "new_error":      round(new_err, 1),
            "reason":         reason,
        })

    except Exception as e:
        print(f"  ❌ {e}")
        log.append({"file": pdf_path, "status": "failed", "original_error": err_pct, "error": str(e)})

# ── Save log ──────────────────────────────────────────────
log_path = Path(REOCR_OUTPUT_DIR) / "_reocr_rerun_log.json"
log_path.write_text(json.dumps(log, ensure_ascii=False, indent=2))

still_bad = sum(1 for r in log if r.get("new_error",999) > REOCR_THRESHOLD)

print(f"\n{'='*55}")
print(f"Re-OCR + Re-LLM Complete")
print(f"  Improved (error dropped >1%)  : {improved}")
print(f"  Same                          : {same}")
print(f"  Worse                         : {worse}")
print(f"  Failed                        : {sum(1 for r in log if r.get('status')=='failed')}")
print(f"  Still > {REOCR_THRESHOLD}%             : {still_bad}")
print(f"  Log → {log_path}")